In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Keyed presence T40–T49 V1

Run all once. Nine independent OFF sources calibrate FULL-only C2 before four new evaluation sources. Each evaluation source generates OFF and the four fixed tanh timing arms; the sole target is keyed H0/H1 presence for T40–T49. ORIGINAL/C1 and terminal-to-MP4 partitions are diagnostics. This notebook does not run automatically and does not establish video quality, identity, payload, low FPR, or extra multistep gain.

In [ ]:
from pathlib import Path
import datetime, json, sys
SOURCE_SHA = '4c4bfb06511d4d1527057a6d781cdd540afe7d06'
DRIVE_ROOT = Path('/content/drive/MyDrive/Video-WM/Keyed-Presence-T40-T49-V1')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT = DRIVE_ROOT / ('keyed_presence_t40_t49_v1_' + stamp)
OUTPUT.mkdir(exist_ok=False)
(OUTPUT / 'setup_receipt.json').write_text(json.dumps(dict(source_commit=SOURCE_SHA, python=sys.version, executable=sys.executable, status='SETUP_STARTED'), indent=2))
print('same-run output:', OUTPUT, flush=True)


In [ ]:
import subprocess, sys
SETUP_LOG = OUTPUT / 'setup.log'
def logged_run(command, check=True, cwd=None, env=None):
    with SETUP_LOG.open('a', encoding='utf-8') as log:
        line = 'COMMAND ' + repr(command) + '\n'
        print(line, end='', flush=True); log.write(line); log.flush()
        child = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in child.stdout:
            print(line, end='', flush=True); log.write(line); log.flush()
        returncode = child.wait()
        log.write('EXIT ' + str(returncode) + '\n'); log.flush()
    if check and returncode:
        (OUTPUT / 'setup_failure.json').write_text(json.dumps(dict(command=command, returncode=returncode), indent=2))
        raise subprocess.CalledProcessError(returncode, command)
    return subprocess.CompletedProcess(command, returncode)
print('Python:', sys.version, 'Executable:', sys.executable, flush=True)
import importlib.metadata, subprocess, sys
print('Python:', sys.version, flush=True)
print('Executable:', sys.executable, flush=True)
logged_run([sys.executable, '-m', 'pip', '--version'], check=True)
logged_run(['apt-get', 'update', '-qq'], check=True)
logged_run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)
def version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None
print('torch before install:', version('torch'), flush=True)
if version('torch') != '2.11.0+cu128':
    logged_run([sys.executable, '-m', 'pip', 'install', 'torch==2.11.0', 'torchvision', '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
logged_run([sys.executable, '-m', 'pip', 'install', 'diffusers==0.40.0', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'], check=True)
check_code = """
import importlib.metadata, sys, torch, diffusers
print('Fresh process Python:', sys.version, flush=True)
print('Fresh process executable:', sys.executable, flush=True)
for name in ('torch', 'torchvision', 'diffusers', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'):
    try:
        value = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        value = None
    print(name + ':', value, flush=True)
assert str(torch.__version__) == '2.11.0+cu128', torch.__version__
assert diffusers.__version__ == '0.40.0', diffusers.__version__
"""
logged_run([sys.executable, "-u", "-c", check_code], check=True)


In [ ]:
import subprocess
REPO = Path('/content/SC-SSTW-Keyed-Presence-' + stamp)
logged_run(['git', 'clone', '--filter=blob:none', 'https://github.com/RICHAAARC/SC-SSTW.git', str(REPO)])
logged_run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_SHA])
actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert actual == SOURCE_SHA
print('source commit:', actual, flush=True)


In [ ]:
import subprocess, sys
check_code = """
import torch
print('torch:', torch.__version__, 'cuda:', torch.version.cuda, flush=True)
if not torch.cuda.is_available():
    raise RuntimeError('This fixed real Wan run requires a CUDA runtime')
print('device:', torch.cuda.get_device_name(0), flush=True)
"""
logged_run([sys.executable, "-u", "-c", check_code], check=True)


In [ ]:
import os
CONFIG = REPO / 'experiments/wan_state_clock/configs/keyed_presence_t40_t49_v1.json'
env = os.environ.copy(); env['PYTHONUNBUFFERED'] = '1'
command = [sys.executable, '-u', '-m', 'experiments.wan_state_clock.keyed_presence_t40_t49_run', '--config', str(CONFIG), '--output', str(OUTPUT)]
print('fixed experiment output:', OUTPUT, flush=True)
logged_run(command, cwd=REPO, env=env)


In [ ]:
import json
result = json.loads((OUTPUT / 'result.json').read_text())
print('status:', result['status'])
print('fixed denominator:', result['fixed_denominator'])
print('C2 calibration:', result['calibration']['C2_STATE_CONFIRM']['status'], result['calibration']['C2_STATE_CONFIRM']['threshold'])
print('target H0/H1:', result['target_summary'])
for case_id, row in result['cases'].items():
    print(case_id, row['status'], {arm: item['receivers']['C2_STATE_CONFIRM'].get('decision') for arm, item in row['videos'].items()})
print('full result:', OUTPUT / 'result.json')
